# Mobile restraints

<video controls src="./assets/mobile_restraints.webm">

## Setup runner & utilities

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../../notebooks/systems/openmm/17-ala.xml")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: moveable restraints")
imd_runner.load(0)

In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

In [3]:
utilities.use_recording_commands()
utilities.selections.update_selection("root", renderer="cartoon")

## Restraints

In [4]:
import numpy as np
from nanover.imd.imd_state import ParticleInteraction
from nanover.jupyter.utilities import make_id_generator

make_new_restraint_key = make_id_generator("restraint.")
MOVEABLE_RESTRAINTS: dict[str, ParticleInteraction] = {}
ACTIVE_RESTRAINTS: set[str] = set()

RESTRAINT_SELECTION_KWARGS = dict(
    renderer={
        "render": {
            "particle.scale": .1,
            "bond.scale": 0.0,
            "type": "ball and stick"
        },
        "color": "CornflowerBlue",
    },
    interaction_method="group",
    velocity_reset=True,
)

RESTRAINT_INTERACTION_KWARGS = dict(
    scale=1000,
    interaction_type="spring",
    is_restraint=True,
)


def add_restraint(indexes):
    key = make_new_restraint_key()

    indexes = [int(index) for index in indexes]

    MOVEABLE_RESTRAINTS[key] = ParticleInteraction(particles=indexes, **RESTRAINT_INTERACTION_KWARGS)
    utilities.selections.update_selection(key, particle_ids=indexes, **RESTRAINT_SELECTION_KWARGS)

    enable_restraint(key)

    return key


def remove_restraint(key: str):
    disable_restraint(key)
    MOVEABLE_RESTRAINTS.pop(key, None)
    utilities.selections.remove_selection(key)
    utilities.objects.remove_shape(key)
    utilities.objects.remove_line(key)


def enable_restraint(key: str):
    if key in ACTIVE_RESTRAINTS:
        return
    ACTIVE_RESTRAINTS.add(key)
    positions = imd_runner.app_server.frame_publisher.current_frame.particle_positions
    restraint = MOVEABLE_RESTRAINTS[key]
    restraint.position = np.mean(positions[restraint.particles], axis=0)
    utilities.interactions.update_interaction(key, restraint)


def disable_restraint(key: str):
    if key not in ACTIVE_RESTRAINTS:
        return
    ACTIVE_RESTRAINTS.discard(key)
    utilities.interactions.remove_interaction(key)


def clear_restraints():
    for restraint in list(MOVEABLE_RESTRAINTS.keys()):
        remove_restraint(restraint)
    MOVEABLE_RESTRAINTS.clear()


def get_associated_restraints(indexes):
    interacted_indexes = set(indexes)
    for key, restraint in MOVEABLE_RESTRAINTS.items():
        if interacted_indexes.intersection(restraint.particles):
            yield key

Add starting restraints:

In [5]:
from nanover.mdanalysis import frame_data_to_mdanalysis

universe = frame_data_to_mdanalysis(simulation.make_topology_frame())
add_restraint(universe.residues[0].atoms.indices), add_restraint(universe.residues[-1].atoms.indices)

('restraint.0', 'restraint.1')

## Visuals

Visualise pinned centroid and draw a line to the actual particle centroid for each restraint:

In [6]:
import numpy as np
from nanover.trajectory import FrameData
from nanover.jupyter import FrameListener


class MobileRestraintVisuals(FrameListener):
    def on_frame_update(self, full_frame: FrameData, frame_update: FrameData):
        with utilities.objects as objects:
            for key, restraint in MOVEABLE_RESTRAINTS.items():
                centroid = np.mean(full_frame.particle_positions[restraint.particles], axis=0)
                distance = np.linalg.norm(centroid - restraint.position)
                strain = min(distance / .5, 1.0)
                color = [1.0, 1.0 - strain, 1.0 - strain, 1.0]
                objects.update_shape(key, position=restraint.position, scale=.5, color=color)
                objects.update_line(key, positions=[restraint.position, centroid], scale=.1, color=color)


visuals = MobileRestraintVisuals.from_runner(imd_runner)
visuals.start()

## Interaction & commands

Command to clear all restraints:

In [7]:
utilities.define_command("user/restraints/clear", handler=clear_restraints, label="clear restraints", icon="🚫")

Default mode disables a restraint while any part of it is being interacted with:

In [8]:
from nanover.jupyter import Mode


class InteractMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        if "is_restraint" in interaction.properties:
            return

        for restraint in get_associated_restraints(interaction.particles):
            disable_restraint(restraint)

    def on_interaction_stopped(self, *, key: str, interaction: ParticleInteraction):
        if "is_restraint" in interaction.properties:
            return

        for restraint in get_associated_restraints(interaction.particles):
            enable_restraint(restraint)


utilities.modes.add_mode(InteractMode(), "normal")
utilities.modes.enter_mode("normal")

Mode that adds/removes a restraint whenever an interaction begins:

In [9]:
from nanover.jupyter import Mode


class ToggleMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        if "is_restraint" in interaction.properties:
            return

        # remove all restraints containing any of these atoms
        restraints = set(get_associated_restraints(interaction.particles))
        for restraint in restraints:
            remove_restraint(restraint)
        # otherwise, add the particles to a new restraint
        if not restraints:
            add_restraint(interaction.particles)


utilities.modes.add_mode(ToggleMode(), "toggle", icon="⛓️")